In [1]:
# base libraries
import sqlite3
import pandas as pd
from pathlib import Path
import os
from difflib import SequenceMatcher
import geonamescache

# NLP 
import spacy
from transformers import pipeline

# topic modeling
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer


In [8]:
# ============================================================
# 0. GLOBAL CONFIG
# ============================================================

DB_PATH = "gnews_articles_from2023.db"
RAW_TABLE = "articles"
ENRICHED_TABLE = "enriched_articles"
OUTPUT_ARTICLES = "processed_conflict_articles.csv"

OUTPUT_EVENTS = "processed_conflict_events.csv"
SIMILARITY_THRESHOLD = 0.7

In [3]:
# ============================================================
# 1. GERMANY LOCATION DICTIONARY
# ============================================================
gc = geonamescache.GeonamesCache()
cities = gc.get_cities()

GERMAN_LOCATIONS = {
    city["name"].lower()
    for city in cities.values()
    if city["countrycode"] == "DE"
}

GERMAN_EXTRAS = {
    "deutschland", "germany", "bundesrepublik", "berlin", "münchen", "munich", 
    "hamburg", "köln", "cologne", "frankfurt", "stuttgart", "düsseldorf", 
    "dortmund", "essen", "leipzig", "bremen", "dresden", "hannover", "nürnberg", 
    "duisburg", "bochum", "wuppertal", "bielefeld", "bonn", "münster", "karlsruhe",
    "mannheim", "augsburg", "wiesbaden", "gelsenkirchen", "mönchengladbach", 
    "braunschweig", "chemnitz", "aachen", "kiel", "halle", "magdeburg", "freiburg", 
    "oberhausen", "lübeck", "erfurt", "mainz", "rostock", "kassel", "hagen", 
    "saarbrücken", "hamm", "potsdam", "ludwigshafen", "oldenburg", "leverkusen", 
    "osnabrück", "solingen", "heidelberg", "herne", "neuss", "darmstadt", "paderborn", 
    "remscheid", "regensburg", "ingolstadt", "würzburg", "wolfsburg", "fürth", "ulm", "offenbach"
}
GERMAN_LOCATIONS.update(GERMAN_EXTRAS)

GERMAN_SUFFIXES = ["straße", "strasse", "platz", "tor", "allee", "bahnhof", "weg", "brücke"]

In [4]:
# ============================================================
# 2. MODEL INITIALIZATION
# ============================================================
print("Initializing NLP Models... (This may take a few minutes)")

try:
    nlp = spacy.load("de_core_news_sm")
except:
    os.system("python -m spacy download de_core_news_sm")
    nlp = spacy.load("de_core_news_sm")

# Multilingual Sentiment (Supports DE, results in Positive/Negative/Neutral)
# This model supports German and outputs: anger, joy, optimism, sadness
emotion_pipe = pipeline(
    "text-classification",
    model="ChrisLalk/German-Emotions",  # or local model path
    tokenizer="ChrisLalk/German-Emotions",
    return_all_scores=True,
    truncation=True,
    top_k=None
)

# Multilingual Zero-Shot (Processes German, outputs English labels)
zero_shot = pipeline("zero-shot-classification", model="MoritzLaurer/mDeBERTa-v3-base-mnli-xnli")




Initializing NLP Models... (This may take a few minutes)


Device set to use mps:0
Device set to use mps:0


In [5]:
# ============================================================
# 3. CORE LOGIC FUNCTIONS
# ============================================================

def detect_domestic_hierarchical(text_de):
    if not text_de: return False, ""
    doc = nlp(text_de)
    found_locs = [ent.text.lower() for ent in doc.ents if ent.label_ in ["GPE", "LOC"]]
    is_domestic = any(loc in GERMAN_LOCATIONS for loc in found_locs)
    if not is_domestic:
        is_domestic = any(any(s in loc for s in GERMAN_SUFFIXES) for loc in found_locs)
        if not is_domestic:
            is_domestic = any(kw in text_de.lower() for kw in GERMAN_EXTRAS)
    return is_domestic, ", ".join(set(found_locs))

def title_similarity(a, b):
    return SequenceMatcher(None, a, b).ratio()

def get_multilingual_emotion(text_de):
    """
    Processes German text using the ChrisLalk Emotion model.
    Maps: joy/surprise -> positive, anger/fear/disgust/sadness -> negative.
    """
    try:
        # Pipeline returns a list of dictionaries with scores for each label
        results = emotion_pipe(text_de[:512])[0]
        # Find the label with the highest score
        top_result = max(results, key=lambda x: x['score'])
        label = top_result['label'].lower()
        score = top_result['score']
        
        # Numeric Mapping for the dashboard's Trend analysis
        if label in ['joy', 'surprise']:
            val = score
        elif label in ['anger', 'fear', 'disgust', 'sadness']:
            val = -score
        else:
            val = 0.0
            
        return label, val
    except:
        return "neutral", 0.0



In [6]:
# ============================================================
# 4. EXECUTION PIPELINE
# ============================================================

def run_enrichment():
    conn = sqlite3.connect(DB_PATH)
    df = pd.read_sql_query(f"SELECT * FROM {RAW_TABLE} ", conn)
    
    df['published_at_dt'] = pd.to_datetime(df['publishedAt'])
    df['published_date'] = df['published_at_dt'].dt.date.astype(str)
    df['published_month'] = df['published_at_dt'].dt.to_period('M').astype(str)

    print("Step A: Geographic Detection...")
    geo_res = df.apply(lambda x: detect_domestic_hierarchical(f"{x['title']} {x['description']}"), axis=1)
    df['is_domestic'] = geo_res.apply(lambda x: x[0])
    df['detected_locations'] = geo_res.apply(lambda x: x[1])

    print("Step B: Clustering...")
    df = df.sort_values(['published_date', 'title'])
    df['article_cluster_id'] = range(len(df))
    df['is_duplicate'] = False
    for i in range(1, len(df)):
        if df.iloc[i]['published_date'] == df.iloc[i-1]['published_date']:
            if title_similarity(df.iloc[i]['title'], df.iloc[i-1]['title']) >= SIMILARITY_THRESHOLD:
                df.iloc[i, df.columns.get_loc('is_duplicate')] = True
                df.iloc[i, df.columns.get_loc('article_cluster_id')] = df.iloc[i-1]['article_cluster_id']

    print("Step C: Event-level Processing...")
    df_events = df.sort_values(by=["article_cluster_id", "published_at_dt"]).groupby("article_cluster_id").first().reset_index()

    # Native German Analysis
    print(f"Analyzing {len(df_events)} events for German emotions...")
    sent_res = df_events.apply(lambda x: get_multilingual_emotion(f"{x['title']} {x['description']}"), axis=1)
    df_events['emotion_label'] = sent_res.apply(lambda x: x[0])
    df_events['sentiment_numeric'] = sent_res.apply(lambda x: x[1])

    acled_labels = ["Protests", "Battles", "Strategic developments", "Violence against civilians", "Riots", "Explosions"]
    df_events['acled_event_type'] = df_events.apply(
        lambda x: zero_shot(f"{x['title']} {x['description']}"[:512], acled_labels)['labels'][0], axis=1
    )

    print("Step D: BERTopic (Multilingual mode)...")
    topic_model = BERTopic(language="multilingual", vectorizer_model=CountVectorizer(stop_words=None, min_df=3))
    topics, _ = topic_model.fit_transform((df_events['title'] + " " + df_events['description'].fillna("")).tolist())
    df_events['narrative_topic_id'] = topics

    print("Step E: Mapping results back...")
    nlp_results = df_events[['article_cluster_id', 'emotion_label', 'sentiment_numeric', 'acled_event_type', 'narrative_topic_id']]
    df_all_enriched = df.merge(nlp_results, on='article_cluster_id', how='left')

    # Storage
    if "publishedAt" in df_all_enriched.columns: df_all_enriched.drop(columns=["publishedAt"], inplace=True)
    df_all_enriched.to_sql(ENRICHED_TABLE, conn, if_exists='replace', index=False)
    df_all_enriched.to_csv(OUTPUT_ARTICLES, index=False, encoding='utf-8-sig')
    df_events.to_csv(OUTPUT_EVENTS, index=False, encoding='utf-8-sig')
    conn.close()
    print("✅ Enrichment and event-reduction complete!")


In [9]:
if __name__ == "__main__":
    run_enrichment()

Step A: Geographic Detection...
Step B: Clustering...


KeyboardInterrupt: 